# Ada in the loop — Qwen3 frontier test

Baseline a real Qwen3, then add Ada's schema **in the loop** (dual objective:
next-token LM + Ada-role token classification) and see if it holds at scale.

**Runtime → Change runtime type → GPU.** A100 (Pro+) runs Qwen3-8B; L4 (Pro) ~4B;
T4 (free) ~1.7B. The script auto-picks the biggest that fits.

Optional: add a HuggingFace token as a Colab secret named `HF_TOKEN`.

In [ ]:
# 1. get the code
!git clone -b claude/ada3-status-ppr17e https://github.com/timmetim/thinktank.git
%cd thinktank/frontier

In [ ]:
# 2. deps + GPU check
!pip install -q transformers peft accelerate bitsandbytes datasets
import torch; print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 3. build data from Ada's real enriched facts (auto-clones glyphh-ai/adaL)
!python prepare_data.py --n 50000

In [ ]:
# 4. CPU smoke test — verify the dual-objective pipeline before GPU spend
!python smoke_test.py

In [ ]:
# 5. ARM B — Ada in the loop (lambda>0) vs plain LM (lambda=0)
#    same model/data/steps; only the role objective differs.
!python finetune_qwen.py --steps 1500 --lam 0.5   # Ada in the loop
!python finetune_qwen.py --steps 1500 --lam 0.0   # plain-LM baseline

## Read the result
- **role-tag accuracy** with lambda=0.5 vs ~chance with lambda=0 → the structure is learned at frontier scale.
- **LM loss** lambda=0.5 ≈ lambda=0 → still free.

Arm A (Qwen3 + Ada store vs RAG, long-horizon recall) extends `benchmark/` in
the adaL repo — swap Qwen3 in as the answerer. See `frontier/README.md`.